# Push-T full-state BC teacher, run 2: more expert data + DAgger

Run 1 trained BC on the 761 released demos that replayed and reached **19% success-once / 10% success-at-end**
on test seeds, versus **100% / 72%** for the released PPO expert. Its training error kept falling while validation
error stalled, a sign of too little data plus compounding errors. See `state_bc/README.md` for the full log.

This run changes only the **training data**. The teacher architecture, loss, optimizer and evaluation are unchanged.
1. **More expert data:** roll out the released PPO expert for 8,192 episodes, some with small action noise so the
   data includes slightly off-course states. Labels are always the expert's clean action.
2. **DAgger:** let the BC policy drive, label every state it reaches with the expert's action, add to the data,
   retrain. Four rounds.

| Setting | Value |
|---|---|
| Observation / controller | `obs_mode="state"` (31 numbers), `pd_joint_delta_pos` (7 numbers) |
| Teacher | MLP, 2 × 256 ReLU, tanh output, MSE, Adam 3e-4 with cosine decay, batch 1024, 100k iterations per training |
| Simulator | ManiSkill v3.0.1 (commit `a4a4f92`), GPU physics |
| Seeds | collection from 100,000+ (expert) and 200,000+ (DAgger); validation 1001; test 5001 (used once, at the end) |

Select **Runtime → Change runtime type → L4 GPU**, then run the cells in order. Upload only this notebook.
Outputs go to `MyDrive/182_final_project/state_bc/<session>/`. Expected time: roughly 35–50 minutes, mostly training.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, datetime, shlex, shutil, urllib.request

MANISKILL_COMMIT = "a4a4f9272ad64b1564035874b605ceb687b63ed8"  # v3.0.1, same as the camera PPO notebook
MS_ROOT = Path("/content/ManiSkill")
WORK_ROOT = Path("/content/pusht_state_bc")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
USE_GOOGLE_DRIVE = True
DRIVE_SUBDIR = "182_final_project/state_bc"
RUN_LABEL = "run2_dagger"

# Teacher training (unchanged from run 1)
TEACHER_NAME = "bc_dagger_seed0"
SEED = 0
SPLIT_SEED = 0
VAL_FRACTION = 0.1
ITERS = 100_000
BATCH_SIZE = 1024
LR = 3e-4
EVAL_EVERY = 25_000           # closed-loop validation every N iterations (run 1 used 10,000)
EVAL_EPISODES = 100
VAL_SEED = 1001
TEST_SEED = 5001

# New in run 2: data collection
NUM_ENVS = 1024
EXPERT_EPISODES = 8192                  # 8 batches of 1024
EXPERT_NOISE = [0.0, 0.1, 0.2, 0.3]     # action-noise std per batch, cycled; labels stay clean
EXPERT_SEED_BASE = 100_000
DAGGER_ROUNDS = 4
DAGGER_EPISODES = 2048                  # per round
DAGGER_SEED_BASE = 200_000              # round r uses DAGGER_SEED_BASE + 1000 * r

DEMO_BASE = "https://huggingface.co/datasets/haosulab/ManiSkill_Demonstrations/resolve/main/demos/PushT-v1/rl/"
DEMO_NAME = "trajectory.none.pd_joint_delta_pos.physx_cuda"
PPO_CKPT_NAME = "ppo_pd_joint_delta_pos_ckpt.pt"
SESSION = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + RUN_LABEL
subprocess.run(["nvidia-smi"], check=True)
print("Notebook Python:", sys.version)

## 1. Save outputs to Drive

Each session gets a new folder, so reruns never overwrite earlier results.

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive") / DRIVE_SUBDIR / SESSION
else:
    OUTPUT_ROOT = WORK_ROOT / "outputs" / SESSION
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
LOG = OUTPUT_ROOT / "notebook.log"
print("Output directory:", OUTPUT_ROOT)


def run(command, **kwargs):
    """Run a command, stream its output here, append it to notebook.log, and raise on failure."""
    command = list(map(str, command))
    print("$", shlex.join(command), flush=True)
    tail = []
    with LOG.open("a") as log:
        log.write("\n$ " + shlex.join(command) + "\n")
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, **kwargs) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    tail = (tail + [line.rstrip()])[-40:]
                code = process.wait()
            except BaseException:
                process.kill()
                raise
    if code:
        raise RuntimeError(f"Command failed (exit {code}). Log: {LOG}\n" + "\n".join(tail))

## 2. Install ManiSkill v3.0.1 under Python 3.11

Same pinned install as run 1 and the camera PPO notebook. Skips work already done in this Colab session.

In [ ]:
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "libvulkan1", "libvulkan-dev", "vulkan-tools", "libglvnd-dev"])
if not (MS_ROOT / ".git").exists():
    run(["git", "clone", "https://github.com/mani-skill/ManiSkill.git", MS_ROOT])
run(["git", "-C", MS_ROOT, "checkout", "--detach", MANISKILL_COMMIT])
assert subprocess.check_output(["git", "-C", str(MS_ROOT), "rev-parse", "HEAD"], text=True).strip() == MANISKILL_COMMIT

run([sys.executable, "-m", "pip", "install", "-q", "uv==0.9.11"])
TRAIN_ENV = WORK_ROOT / "python311"
TRAIN_PY = TRAIN_ENV / "bin/python"
if not TRAIN_PY.exists():
    run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "--seed", TRAIN_ENV])
constraints = WORK_ROOT / "constraints.txt"
constraints.write_text("numpy==1.26.4\nopencv-python==4.11.0.86\nsapien==3.0.3\ntorch==2.6.0\n")
run([TRAIN_PY, "-m", "pip", "install", "-q", "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"])
run([TRAIN_PY, "-m", "pip", "install", "-q", "--constraint", constraints, "-e", MS_ROOT,
     "numpy", "sapien", "h5py", "tyro", "imageio[ffmpeg]", "matplotlib"])

# Vulkan/EGL entries Colab is missing; needed for rendering evaluation videos.
for filename, library, vulkan in [
    ("/usr/share/vulkan/icd.d/nvidia_icd.json", "libGLX_nvidia.so.0", True),
    ("/usr/share/glvnd/egl_vendor.d/10_nvidia.json", "libEGL_nvidia.so.0", False),
]:
    path = Path(filename)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        icd = {"library_path": library}
        if vulkan:
            icd["api_version"] = "1.2.155"
        path.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": icd}))
os.environ["VK_ICD_FILENAMES"] = "/usr/share/vulkan/icd.d/nvidia_icd.json"
os.environ["MS_SKIP_ASSET_DOWNLOAD_PROMPT"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
run([TRAIN_PY, "-c", "import torch, sapien, mani_skill, numpy; print('torch', torch.__version__, 'CUDA', torch.version.cuda); "
     "print('mani_skill', mani_skill.__version__); assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"])
(OUTPUT_ROOT / "pip-freeze.txt").write_text(subprocess.check_output([str(TRAIN_PY), "-m", "pip", "freeze"], text=True))
print("Training Python:", TRAIN_PY)

## 3. Write the BC module

`pusht_state_bc.py` holds all code: `audit`, `prepare`, `collect`, `train`, `eval`.
The repo copy is `state_bc/pusht_state_bc.py`; this notebook is generated from it by `state_bc/build_notebook.py`.

In [ ]:
%%writefile /content/pusht_state_bc/pusht_state_bc.py
"""Full-state behavioral cloning (BC) teachers for ManiSkill PushT-v1.

Subcommands (run with the ManiSkill training interpreter):

    audit    Inspect the state observation and action spaces; write observation_contract.json.
    prepare  Convert a replayed state-mode demo file (.h5 + .json) into train/validation
             transition files, split by whole episode.
    collect  Roll out a driver policy (the expert, or a BC checkpoint for DAgger) and label
             every visited state with the expert's deterministic action.
    train    Train one MLP BC teacher on one or more transition files, evaluating closed-loop
             success on validation seeds.
    eval     Evaluate a BC checkpoint, or the released PPO expert, on a seed range.

Transition files are .npz with obs [N, d_s], actions [N, d_a] and episode_ids [N],
plus a .json sidecar describing where they came from.

Pure data/model code is importable without ManiSkill, so it can be unit tested offline.
ManiSkill and gymnasium are imported lazily inside the simulator functions.
"""

from __future__ import annotations

import argparse
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn

ENV_ID = "PushT-v1"
OBS_MODE = "state"
CONTROL_MODE = "pd_joint_delta_pos"
SIM_BACKEND = "physx_cuda"
# Features whose training std is below this are only centered, not scaled. Near-constant
# fields (for example the fixed goal position) would otherwise turn simulator noise
# into very large normalized values.
MIN_SCALED_STD = 1e-3


# --------------------------------------------------------------------------------------
# Transition files
# --------------------------------------------------------------------------------------

def save_transitions(path, obs, actions, episode_ids, meta=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, obs=np.asarray(obs, np.float32), actions=np.asarray(actions, np.float32),
                        episode_ids=np.asarray(episode_ids, np.int64))
    meta = dict(meta or {}, transitions=int(len(obs)), episodes=int(len(np.unique(episode_ids))))
    path.with_suffix(".json").write_text(json.dumps(meta, indent=2))
    return meta


def load_transitions(paths):
    """Concatenate transition files; episode ids are made unique across files."""
    obs, actions, episode_ids, offset = [], [], [], 0
    for p in paths:
        d = np.load(p)
        obs.append(d["obs"])
        actions.append(d["actions"])
        ids = d["episode_ids"].astype(np.int64)
        episode_ids.append(ids - ids.min() + offset if len(ids) else ids)
        offset = (episode_ids[-1].max() + 1) if len(ids) else offset
    return np.concatenate(obs), np.concatenate(actions), np.concatenate(episode_ids)


# --------------------------------------------------------------------------------------
# Released demos
# --------------------------------------------------------------------------------------

def load_replayed_demos(h5_path: str | Path):
    """Return a list of episodes, each a dict with obs [T, d_s], actions [T, d_a], and metadata.

    The replayed file stores T+1 observations per episode; the last one has no action
    and is dropped, as in ManiSkill's own BC baseline.
    """
    import h5py

    h5_path = Path(h5_path)
    meta = json.loads(h5_path.with_suffix(".json").read_text())
    env_kwargs = meta["env_info"]["env_kwargs"]
    if env_kwargs.get("control_mode") not in (None, CONTROL_MODE):
        raise ValueError(f"Demo control mode {env_kwargs.get('control_mode')} != {CONTROL_MODE}")
    if env_kwargs.get("obs_mode") != OBS_MODE:
        raise ValueError(f"Demo obs mode {env_kwargs.get('obs_mode')} != {OBS_MODE}; replay with -o state")
    episodes = []
    with h5py.File(h5_path, "r") as f:
        for ep in meta["episodes"]:
            group = f[f"traj_{ep['episode_id']}"]
            obs = np.asarray(group["obs"], dtype=np.float32)
            actions = np.asarray(group["actions"], dtype=np.float32)
            if obs.ndim != 2 or actions.ndim != 2 or obs.shape[0] != actions.shape[0] + 1:
                raise ValueError(f"Unexpected shapes in traj_{ep['episode_id']}: obs {obs.shape}, actions {actions.shape}")
            success = np.asarray(group["success"]) if "success" in group else None
            episodes.append(dict(
                episode_id=int(ep["episode_id"]),
                episode_seed=ep.get("episode_seed"),
                obs=obs[:-1],
                # The simulator clips commands to the action space, so clipped labels are
                # exactly what the expert executed.
                actions=np.clip(actions, -1.0, 1.0),
                raw_actions=actions,
                success_at_end=bool(success[-1]) if success is not None else bool(ep.get("success", False)),
            ))
    return episodes, meta


def split_episodes(num_episodes: int, val_fraction: float, seed: int):
    """Split episode indices into train/validation sets, whole episodes only."""
    if not 0 < val_fraction < 1:
        raise ValueError("val_fraction must be in (0, 1)")
    rng = np.random.default_rng(seed)
    order = rng.permutation(num_episodes)
    num_val = max(1, int(round(num_episodes * val_fraction)))
    return np.sort(order[num_val:]), np.sort(order[:num_val])


def compute_state_stats(states: np.ndarray):
    mean = states.mean(axis=0)
    std = states.std(axis=0)
    scale = np.where(std < MIN_SCALED_STD, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32), scale.astype(np.float32)


def prepare(args):
    episodes, meta = load_replayed_demos(args.h5)
    if args.max_episodes:
        episodes = episodes[:args.max_episodes]
    train_idx, val_idx = split_episodes(len(episodes), args.val_fraction, args.split_seed)
    out = Path(args.out_dir)
    source = dict(source="released PPO demos, replayed to state obs", source_h5=str(args.h5),
                  source_env_info=meta["env_info"], source_commit=meta.get("commit_info"),
                  split_seed=args.split_seed, val_fraction=args.val_fraction)
    for name, idx in (("demos_train", train_idx), ("demos_val", val_idx)):
        eps = [episodes[i] for i in idx]
        save_transitions(out / f"{name}.npz",
                         np.concatenate([e["obs"] for e in eps]),
                         np.concatenate([e["actions"] for e in eps]),
                         np.concatenate([np.full(len(e["actions"]), e["episode_id"]) for e in eps]),
                         dict(source, split=name, episode_ids=[e["episode_id"] for e in eps]))
    raw = np.concatenate([e["raw_actions"] for e in episodes])
    summary = dict(
        state_dim=int(episodes[0]["obs"].shape[1]), action_dim=int(raw.shape[1]),
        episodes_in_file=len(meta["episodes"]), episodes_replayed=len(episodes),
        train_episodes=len(train_idx), val_episodes=len(val_idx),
        raw_action_min=raw.min(0).round(4).tolist(), raw_action_max=raw.max(0).round(4).tolist(),
        fraction_raw_actions_outside_bounds=float((np.abs(raw) > 1).mean()),
        replayed_success_at_end=float(np.mean([e["success_at_end"] for e in episodes])),
    )
    (out / "demos_summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))


# --------------------------------------------------------------------------------------
# Models
# --------------------------------------------------------------------------------------

class BCPolicy(nn.Module):
    """MLP teacher: normalized state -> action in [-1, 1].

    Normalization statistics are registered buffers, so a checkpoint carries its own
    preprocessing and the policy accepts raw environment observations.
    """

    def __init__(self, state_dim, action_dim, state_mean, state_scale, hidden=256, depth=2, tanh_output=True):
        super().__init__()
        layers, width = [], state_dim
        for _ in range(depth):
            layers += [nn.Linear(width, hidden), nn.ReLU()]
            width = hidden
        layers.append(nn.Linear(width, action_dim))
        self.net = nn.Sequential(*layers)
        self.tanh_output = tanh_output
        self.register_buffer("state_mean", torch.as_tensor(state_mean, dtype=torch.float32))
        self.register_buffer("state_scale", torch.as_tensor(state_scale, dtype=torch.float32))
        self.config = dict(state_dim=state_dim, action_dim=action_dim, hidden=hidden, depth=depth, tanh_output=tanh_output)

    def forward(self, raw_state):
        out = self.net((raw_state - self.state_mean) / self.state_scale)
        return torch.tanh(out) if self.tanh_output else out

    @torch.no_grad()
    def act(self, raw_state):
        return self(raw_state).clamp(-1.0, 1.0)


def save_policy(policy: BCPolicy, path, extra=None):
    torch.save({"config": policy.config, "state_dict": policy.state_dict(), "extra": extra or {}}, path)


def load_policy(path, device="cpu") -> BCPolicy:
    ckpt = torch.load(path, map_location=device, weights_only=False)
    cfg = ckpt["config"]
    policy = BCPolicy(cfg["state_dim"], cfg["action_dim"], np.zeros(cfg["state_dim"]), np.ones(cfg["state_dim"]),
                      hidden=cfg["hidden"], depth=cfg["depth"], tanh_output=cfg["tanh_output"])
    policy.load_state_dict(ckpt["state_dict"])
    return policy.to(device).eval()


class ReleasedPPOActor(nn.Module):
    """Deterministic mean of ManiSkill's state PPO actor (examples/baselines/ppo/ppo.py)."""

    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.actor_mean = nn.Sequential(
            nn.Linear(state_dim, 256), nn.Tanh(), nn.Linear(256, 256), nn.Tanh(),
            nn.Linear(256, 256), nn.Tanh(), nn.Linear(256, action_dim))

    @torch.no_grad()
    def act(self, raw_state):
        return self.actor_mean(raw_state).clamp(-1.0, 1.0)


def load_released_ppo(path, state_dim, action_dim, device="cpu"):
    weights = torch.load(path, map_location=device, weights_only=False)
    actor = ReleasedPPOActor(state_dim, action_dim)
    actor.load_state_dict({k: v for k, v in weights.items() if k.startswith("actor_mean.")})
    return actor.to(device).eval()


# --------------------------------------------------------------------------------------
# Rollouts (shared by evaluation and data collection)
# --------------------------------------------------------------------------------------

@torch.no_grad()
def rollout(env, max_steps, driver, seed, labeler=None, noise_std=0.0, generator=None):
    """Run one batch of episodes, one per parallel env, for max_steps with no resets.

    driver picks the executed action; Gaussian noise (std on the [-1, 1] action scale) is
    added after labeling, so labels stay the labeler's clean actions. If labeler is given,
    returns obs [n, T, d_s] and labels [n, T, d_a] on the CPU, episode-major.
    """
    obs, _ = env.reset(seed=seed)
    n = obs.shape[0]
    device = obs.device
    success_once = torch.zeros(n, dtype=torch.bool, device=device)
    first_success = torch.full((n,), -1, dtype=torch.long, device=device)
    obs_buf, label_buf = [], []
    success = success_once
    for t in range(max_steps):
        o = obs.float()
        action = driver.act(o)
        if labeler is not None:
            label = action if labeler is driver else labeler.act(o)
            obs_buf.append(o.cpu())
            label_buf.append(label.cpu())
        if noise_std > 0:
            noise = torch.randn(action.shape, device=action.device, generator=generator)
            action = (action + noise_std * noise).clamp(-1.0, 1.0)
        obs, _, _, _, info = env.step(action)
        success = info["success"].bool()
        newly = (first_success < 0) & success
        first_success = torch.where(newly, torch.full_like(first_success, t + 1), first_success)
        success_once |= success
    steps = first_success[first_success > 0].float()
    result = {
        "episodes": int(n), "seed": int(seed), "noise_std": float(noise_std),
        "success_once": success_once.float().mean().item(),
        "success_at_end": success.float().mean().item(),
        "mean_steps_to_first_success": steps.mean().item() if len(steps) else None,
    }
    if labeler is None:
        return result
    return result, torch.stack(obs_buf, 1).numpy(), torch.stack(label_buf, 1).numpy()


# --------------------------------------------------------------------------------------
# Training
# --------------------------------------------------------------------------------------

@torch.no_grad()
def action_mse(policy, obs, actions, device, batch=65536):
    total, count = 0.0, 0
    for i in range(0, len(obs), batch):
        o = torch.as_tensor(obs[i:i + batch], device=device)
        a = torch.as_tensor(actions[i:i + batch], device=device)
        total += torch.nn.functional.mse_loss(policy(o), a, reduction="sum").item()
        count += a.numel()
    return total / max(count, 1)


def train(args):
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    device = torch.device(args.device)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    obs_np, act_np, ep_np = load_transitions(args.train_data)
    val_obs, val_act, _ = load_transitions([args.val_data])
    mean, std, scale = compute_state_stats(obs_np)
    data_info = dict(train_files=[str(p) for p in args.train_data], val_file=str(args.val_data),
                     train_transitions=int(len(obs_np)), train_episodes=int(len(np.unique(ep_np))),
                     val_transitions=int(len(val_obs)),
                     features_only_centered=np.flatnonzero(std < MIN_SCALED_STD).tolist())
    (out / "data_info.json").write_text(json.dumps(data_info, indent=2))
    print(f"Training on {data_info['train_transitions']:,} transitions from "
          f"{data_info['train_episodes']:,} episodes in {len(args.train_data)} file(s)", flush=True)

    train_obs = torch.as_tensor(obs_np, device=device)
    train_act = torch.as_tensor(act_np, device=device).clamp(-1.0, 1.0)
    if args.tanh_output:
        # tanh cannot reach exactly +-1; keep targets just inside the bound.
        train_act = train_act.clamp(-0.999, 0.999)
    state_dim, action_dim = train_obs.shape[1], train_act.shape[1]
    policy = BCPolicy(state_dim, action_dim, mean, scale,
                      hidden=args.hidden, depth=args.depth, tanh_output=args.tanh_output).to(device)
    opt = torch.optim.Adam(policy.parameters(), lr=args.lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.iters) if args.cosine else None
    gen = torch.Generator(device=device).manual_seed(args.seed)

    evaluator = Evaluator(num_envs=args.eval_episodes, device=device) if args.eval_every > 0 else None

    fields = ("iter", "train_mse", "val_mse", "lr", "success_once", "success_at_end", "seconds")
    log = (out / "train_log.csv").open("w")
    log.write(",".join(fields) + "\n")
    best = (-1.0, -1.0)
    start = time.time()
    running, samples = 0.0, 0
    for it in range(1, args.iters + 1):
        idx = torch.randint(0, len(train_obs), (args.batch_size,), device=device, generator=gen)
        loss = torch.nn.functional.mse_loss(policy(train_obs[idx]), train_act[idx])
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if sched:
            sched.step()
        if it % 100 == 0:
            running += loss.item()
            samples += 1

        last = it == args.iters
        if it % args.log_every == 0 or last:
            policy.eval()
            val_mse = action_mse(policy, val_obs, val_act.clip(-1, 1), device)
            row = dict(iter=it, train_mse=running / max(samples, 1), val_mse=val_mse,
                       lr=opt.param_groups[0]["lr"], success_once="", success_at_end="")
            running, samples = 0.0, 0
            if evaluator and (it % args.eval_every == 0 or last):
                res = evaluator.run(policy, seed=args.val_seed)
                row.update(success_once=res["success_once"], success_at_end=res["success_at_end"])
                save_policy(policy, out / f"ckpt_{it}.pt", {"iter": it, **res})
                key = (res["success_at_end"], res["success_once"])
                if key > best:
                    best = key
                    save_policy(policy, out / "best.pt", {"iter": it, **res})
                    (out / "best.json").write_text(json.dumps({"iter": it, **res}, indent=2))
            policy.train()
            row["seconds"] = round(time.time() - start, 1)
            log.write(",".join(str(row[k]) for k in fields) + "\n")
            log.flush()
            msg = f"iter {it:>7}  train_mse {row['train_mse']:.5f}  val_mse {val_mse:.5f}"
            if row["success_once"] != "":
                msg += f"  val success_once {row['success_once']:.0%}  success_at_end {row['success_at_end']:.0%}"
            print(msg, flush=True)
    log.close()
    save_policy(policy, out / "final.pt", {"iter": args.iters})
    if evaluator is None:
        save_policy(policy, out / "best.pt", {"iter": args.iters, "note": "no closed-loop eval; best = final"})
    else:
        evaluator.close()
    print(f"Done in {(time.time() - start) / 60:.1f} min. Best validation (end, once): {best}")


# --------------------------------------------------------------------------------------
# Simulator: audit, evaluation, collection
# --------------------------------------------------------------------------------------

def make_env(num_envs, obs_mode=OBS_MODE, **kwargs):
    import gymnasium as gym
    import mani_skill.envs  # noqa: F401  (registers environments)

    return gym.make(ENV_ID, num_envs=num_envs, obs_mode=obs_mode, control_mode=CONTROL_MODE,
                    sim_backend=SIM_BACKEND, render_mode="rgb_array", **kwargs)


def episode_length(env):
    """Max episode steps. Vectorized GPU envs leave env.spec.max_episode_steps as None,
    so use ManiSkill's own lookup (as its PPO baseline does), with PushT-v1's registered 100 as fallback."""
    from mani_skill.utils import gym_utils

    steps = gym_utils.find_max_episode_steps_value(env)
    return int(steps) if steps is not None else 100


def make_vector_env(num_envs, video_dir=None, **kwargs):
    """Fixed-length episodes: no auto-reset (it would replace the final step's info) and
    terminations ignored, matching the evaluation convention of ManiSkill's PPO baseline."""
    from mani_skill.vector.wrappers.gymnasium import ManiSkillVectorEnv

    env = make_env(num_envs, **kwargs)
    max_steps = episode_length(env)
    if video_dir is not None:
        from mani_skill.utils.wrappers.record import RecordEpisode
        env = RecordEpisode(env, output_dir=str(video_dir), save_trajectory=False,
                            max_steps_per_video=max_steps, video_fps=30)
    return ManiSkillVectorEnv(env, num_envs, auto_reset=False, ignore_terminations=True, record_metrics=True), max_steps


def flatten_with_names(tree, prefix=""):
    """Flatten a nested state dict in insertion order, as ManiSkill does; return names and widths."""
    fields = []
    for key, value in tree.items():
        name = f"{prefix}{key}"
        if isinstance(value, dict):
            fields += flatten_with_names(value, name + "/")
        else:
            width = 1 if value.ndim == 1 else int(value.shape[-1])
            fields.append((name, width))
    return fields


def audit(args):
    import mani_skill

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    env = make_env(1)
    try:
        obs, info = env.reset(seed=0)
        tree = env.unwrapped.get_obs(info, unflattened=True)
        fields = flatten_with_names(tree)
        width = sum(w for _, w in fields)
        assert obs.shape == (1, width), f"Flat obs {tuple(obs.shape)} does not match fields totalling {width}"
        offsets, start = {}, 0
        for name, w in fields:
            offsets[name] = [start, start + w]
            start += w
        if not any("obj_pose" in n for n, _ in fields):
            raise AssertionError("State observation lacks obj_pose; this is not the full-state setting")
        space = env.unwrapped.single_action_space
        contract = {
            "env_id": ENV_ID, "obs_mode": OBS_MODE, "control_mode": CONTROL_MODE, "sim_backend": SIM_BACKEND,
            "mani_skill_version": getattr(mani_skill, "__version__", "unknown"),
            "state_dim": width, "state_fields_in_order": offsets,
            "action_dim": int(np.prod(space.shape)),
            "action_low": space.low.tolist(), "action_high": space.high.tolist(),
            "control_freq": getattr(env.unwrapped, "control_freq", None),
            "sim_freq": getattr(env.unwrapped, "sim_freq", None),
            "max_episode_steps": episode_length(env),
            "robot_uids": str(getattr(env.unwrapped, "robot_uids", None)),
            "success_rule": "evaluate(): T overlap with goal >= intersection_thresh",
            "intersection_thresh": getattr(env.unwrapped, "intersection_thresh", None),
        }
        if not (np.allclose(space.low, -1) and np.allclose(space.high, 1)):
            print("WARNING: action space is not [-1, 1]; the BC tanh head assumes normalized actions.")
        (out / "observation_contract.json").write_text(json.dumps(contract, indent=2))
        print(json.dumps(contract, indent=2))
    finally:
        env.close()


class Evaluator:
    """Closed-loop evaluation on a fixed batch of seeded initial states."""

    def __init__(self, num_envs, device, video_dir=None):
        self.env, self.max_steps = make_vector_env(num_envs, video_dir=video_dir, reconfiguration_freq=1)
        self.num_envs = num_envs
        self.device = device

    def run(self, policy, seed):
        return rollout(self.env, self.max_steps, policy, seed)

    def close(self):
        self.env.close()


def load_expert(path, env, device):
    state_dim = int(np.prod(env.single_observation_space.shape))
    action_dim = int(np.prod(env.single_action_space.shape))
    return load_released_ppo(path, state_dim, action_dim, device)


def evaluate(args):
    device = torch.device(args.device)
    video_dir = Path(args.video_dir) if args.video_dir else None
    evaluator = Evaluator(args.episodes, device, video_dir=video_dir)
    try:
        if args.released_ppo:
            policy = load_expert(args.checkpoint, evaluator.env, device)
        else:
            policy = load_policy(args.checkpoint, device)
        res = evaluator.run(policy, seed=args.seed)
    finally:
        evaluator.close()
    res["checkpoint"] = str(args.checkpoint)
    print(json.dumps(res, indent=2))
    if args.out:
        Path(args.out).write_text(json.dumps(res, indent=2))
    return res


def collect(args):
    """Roll out a driver and label every visited state with the expert.

    driver=expert (optionally with action noise) gives more, and slightly off-course,
    expert demonstrations. driver=bc gives one round of DAgger data: states the BC policy
    actually reaches, labeled with what the expert would do there.
    """
    device = torch.device(args.device)
    env, max_steps = make_vector_env(args.num_envs)
    try:
        expert = load_expert(args.expert, env, device)
        driver = expert if args.driver == "expert" else load_policy(args.driver_checkpoint, device)
        noise_levels = args.noise_std or [0.0]
        gen = torch.Generator(device=device).manual_seed(args.seed_base)
        batches = math.ceil(args.episodes / args.num_envs)
        all_obs, all_labels, per_batch = [], [], []
        for b in range(batches):
            sigma = noise_levels[b % len(noise_levels)]
            res, obs, labels = rollout(env, max_steps, driver, seed=args.seed_base + b,
                                       labeler=expert, noise_std=sigma, generator=gen)
            all_obs.append(obs)
            all_labels.append(labels)
            per_batch.append(res)
            print(f"batch {b + 1}/{batches}  noise {sigma:.2f}  driver success_once {res['success_once']:.0%}  "
                  f"success_at_end {res['success_at_end']:.0%}", flush=True)
    finally:
        env.close()
    obs = np.concatenate(all_obs)[:args.episodes]       # [episodes, T, d_s]
    labels = np.concatenate(all_labels)[:args.episodes]
    n, T = obs.shape[:2]
    meta = dict(
        driver=args.driver, driver_checkpoint=args.driver_checkpoint or None, expert=str(args.expert),
        labels="expert deterministic action, clipped to [-1, 1]", noise_std_cycle=noise_levels,
        seed_base=args.seed_base, num_envs=args.num_envs, steps_per_episode=int(T), batches=per_batch,
        driver_success_once=float(np.mean([r["success_once"] for r in per_batch])),
        driver_success_at_end=float(np.mean([r["success_at_end"] for r in per_batch])),
    )
    meta = save_transitions(args.out, obs.reshape(n * T, -1), labels.reshape(n * T, -1),
                            np.repeat(np.arange(n), T), meta)
    print(json.dumps({k: v for k, v in meta.items() if k != "batches"}, indent=2))


# --------------------------------------------------------------------------------------
# CLI
# --------------------------------------------------------------------------------------

def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)
    default_device = "cuda" if torch.cuda.is_available() else "cpu"

    a = sub.add_parser("audit")
    a.add_argument("--out", required=True)

    pr = sub.add_parser("prepare")
    pr.add_argument("--h5", required=True)
    pr.add_argument("--out-dir", required=True)
    pr.add_argument("--val-fraction", type=float, default=0.1)
    pr.add_argument("--split-seed", type=int, default=0)
    pr.add_argument("--max-episodes", type=int, default=0, help="0 = use all episodes")

    c = sub.add_parser("collect")
    c.add_argument("--expert", required=True, help="released PPO checkpoint used for labels")
    c.add_argument("--driver", choices=("expert", "bc"), default="expert")
    c.add_argument("--driver-checkpoint", default="", help="BC checkpoint when --driver bc")
    c.add_argument("--episodes", type=int, required=True)
    c.add_argument("--num-envs", type=int, default=1024)
    c.add_argument("--noise-std", type=float, nargs="*", default=[0.0],
                   help="action noise std per batch, cycled; labels stay noise-free")
    c.add_argument("--seed-base", type=int, required=True)
    c.add_argument("--out", required=True)
    c.add_argument("--device", default=default_device)

    t = sub.add_parser("train")
    t.add_argument("--train-data", nargs="+", required=True)
    t.add_argument("--val-data", required=True)
    t.add_argument("--out", required=True)
    t.add_argument("--seed", type=int, default=0)
    t.add_argument("--iters", type=int, default=100_000)
    t.add_argument("--batch-size", type=int, default=1024)
    t.add_argument("--lr", type=float, default=3e-4)
    t.add_argument("--cosine", action=argparse.BooleanOptionalAction, default=True)
    t.add_argument("--hidden", type=int, default=256)
    t.add_argument("--depth", type=int, default=2)
    t.add_argument("--tanh-output", action=argparse.BooleanOptionalAction, default=True)
    t.add_argument("--log-every", type=int, default=1000)
    t.add_argument("--eval-every", type=int, default=10_000, help="0 disables closed-loop eval")
    t.add_argument("--eval-episodes", type=int, default=100)
    t.add_argument("--val-seed", type=int, default=1001)
    t.add_argument("--device", default=default_device)

    e = sub.add_parser("eval")
    e.add_argument("--checkpoint", required=True)
    e.add_argument("--released-ppo", action="store_true", help="checkpoint is ManiSkill's released PPO .pt")
    e.add_argument("--episodes", type=int, default=100)
    e.add_argument("--seed", type=int, default=2001)
    e.add_argument("--out", default="")
    e.add_argument("--video-dir", default="")
    e.add_argument("--device", default=default_device)

    args = p.parse_args(argv)
    if args.cmd == "collect" and args.driver == "bc" and not args.driver_checkpoint:
        p.error("--driver bc needs --driver-checkpoint")
    {"audit": audit, "prepare": prepare, "collect": collect, "train": train, "eval": evaluate}[args.cmd](args)


if __name__ == "__main__":
    main()


In [ ]:
BC = WORK_ROOT / "pusht_state_bc.py"
shutil.copy2(BC, OUTPUT_ROOT / BC.name)
run([TRAIN_PY, BC, "--help"])

## 4. Check the observation and action spaces

Writes `observation_contract.json` (unchanged from run 1).

In [ ]:
run([TRAIN_PY, BC, "audit", "--out", OUTPUT_ROOT])
contract = json.loads((OUTPUT_ROOT / "observation_contract.json").read_text())
assert contract["control_mode"] == "pd_joint_delta_pos" and contract["obs_mode"] == "state"
assert contract["max_episode_steps"] == 100, contract["max_episode_steps"]

## 5. Released demos and expert checkpoint

Same as run 1: download, replay into state observations, split whole episodes 90/10.
The released demos stay in the training data, and their validation split is used for validation action error.

In [ ]:
DEMO_DIR = WORK_ROOT / "demos/PushT-v1/rl"
DEMO_DIR.mkdir(parents=True, exist_ok=True)
for name in (DEMO_NAME + ".h5", DEMO_NAME + ".json", PPO_CKPT_NAME):
    target = DEMO_DIR / name
    if not target.exists():
        print("Downloading", name)
        urllib.request.urlretrieve(DEMO_BASE + name, target)
    print(f"{name}: {target.stat().st_size / 1e6:.1f} MB")
EXPERT = DEMO_DIR / PPO_CKPT_NAME

replayed = DEMO_DIR / "trajectory.state.pd_joint_delta_pos.physx_cuda.h5"
if not replayed.exists():
    run([TRAIN_PY, "-m", "mani_skill.trajectory.replay_trajectory",
         "--traj-path", DEMO_DIR / (DEMO_NAME + ".h5"), "--use-env-states",
         "-c", "pd_joint_delta_pos", "-o", "state", "--save-traj", "--num-envs", 1024, "-b", "physx_cuda"])

DATA_DIR = OUTPUT_ROOT / "data"
run([TRAIN_PY, BC, "prepare", "--h5", replayed, "--out-dir", DATA_DIR,
     "--val-fraction", VAL_FRACTION, "--split-seed", SPLIT_SEED])
DEMOS_TRAIN, DEMOS_VAL = DATA_DIR / "demos_train.npz", DATA_DIR / "demos_val.npz"

## 6. Released PPO expert on validation seeds

Reference point. Run 1 measured **98% success-once / 65% success-at-end** here.

In [ ]:
run([TRAIN_PY, BC, "eval", "--checkpoint", EXPERT, "--released-ppo", "--episodes", EVAL_EPISODES,
     "--seed", VAL_SEED, "--out", OUTPUT_ROOT / "released_ppo_val_eval.json"])

## 7. Collect more expert data (new)

The expert drives 8,192 episodes of 100 steps (about 819k transitions). Batches cycle through action-noise levels
0, 0.1, 0.2 and 0.3, so some episodes wander off the expert's usual path and record how it corrects.
Labels are always the expert's noise-free action. The printed success rates show how much noise the expert tolerates.

In [ ]:
EXPERT_DATA = DATA_DIR / "expert_rollouts.npz"
run([TRAIN_PY, BC, "collect", "--expert", EXPERT, "--driver", "expert", "--episodes", EXPERT_EPISODES,
     "--num-envs", NUM_ENVS, "--noise-std", *EXPERT_NOISE, "--seed-base", EXPERT_SEED_BASE, "--out", EXPERT_DATA])

## 8. Smoke test

Short run on the new data to catch problems before the long jobs.

In [ ]:
run([TRAIN_PY, BC, "train", "--train-data", DEMOS_TRAIN, EXPERT_DATA, "--val-data", DEMOS_VAL,
     "--out", OUTPUT_ROOT / "smoke", "--seed", SEED, "--iters", 2000, "--log-every", 500,
     "--eval-every", 1000, "--eval-episodes", 16, "--val-seed", VAL_SEED])

## 9. Round 0: train on released demos + expert rollouts

This alone measures how much **more expert data** helps, before any DAgger.

In [ ]:
RUNS = OUTPUT_ROOT / "runs" / TEACHER_NAME
config = dict(teacher=TEACHER_NAME, seed=SEED, split_seed=SPLIT_SEED, iters=ITERS, batch_size=BATCH_SIZE, lr=LR,
              eval_every=EVAL_EVERY, eval_episodes=EVAL_EPISODES, val_seed=VAL_SEED, test_seed=TEST_SEED,
              expert_episodes=EXPERT_EPISODES, expert_noise=EXPERT_NOISE, dagger_rounds=DAGGER_ROUNDS,
              dagger_episodes=DAGGER_EPISODES, maniskill_commit=MANISKILL_COMMIT,
              architecture="MLP 2x256 ReLU, tanh output", training="from scratch each round on all data so far")
RUNS.mkdir(parents=True, exist_ok=False)
(RUNS / "run_config.json").write_text(json.dumps(config, indent=2))


def train_round(r, train_files):
    out = RUNS / f"round{r}"
    run([TRAIN_PY, BC, "train", "--train-data", *train_files, "--val-data", DEMOS_VAL, "--out", out,
         "--seed", SEED, "--iters", ITERS, "--batch-size", BATCH_SIZE, "--lr", LR,
         "--eval-every", EVAL_EVERY, "--eval-episodes", EVAL_EPISODES, "--val-seed", VAL_SEED])
    return out


progress = []
train_files = [DEMOS_TRAIN, EXPERT_DATA]
round_dir = train_round(0, train_files)
best = json.loads((round_dir / "best.json").read_text())
progress.append(dict(round=0, train_files=len(train_files), val_success_once=best["success_once"],
                     val_success_at_end=best["success_at_end"], best_iter=best["iter"], on_policy=None))
(RUNS / "dagger_progress.json").write_text(json.dumps(progress, indent=2))
print(progress[-1])

## 10. DAgger rounds (new)

Each round: the latest BC teacher drives 2,048 episodes, the expert labels every state it visits,
those transitions join the training data, and a fresh BC teacher trains on everything so far.
`on_policy` is the driving BC policy's own success while collecting, a second view of progress.

In [ ]:
for r in range(1, DAGGER_ROUNDS + 1):
    previous = RUNS / f"round{r - 1}" / "best.pt"
    dagger_data = DATA_DIR / f"dagger_round{r}.npz"
    run([TRAIN_PY, BC, "collect", "--expert", EXPERT, "--driver", "bc", "--driver-checkpoint", previous,
         "--episodes", DAGGER_EPISODES, "--num-envs", NUM_ENVS, "--seed-base", DAGGER_SEED_BASE + 1000 * r,
         "--out", dagger_data])
    collected = json.loads(dagger_data.with_suffix(".json").read_text())
    train_files.append(dagger_data)
    round_dir = train_round(r, train_files)
    best = json.loads((round_dir / "best.json").read_text())
    progress.append(dict(round=r, train_files=len(train_files), val_success_once=best["success_once"],
                         val_success_at_end=best["success_at_end"], best_iter=best["iter"],
                         on_policy=dict(success_once=collected["driver_success_once"],
                                        success_at_end=collected["driver_success_at_end"])))
    (RUNS / "dagger_progress.json").write_text(json.dumps(progress, indent=2))
    print(progress[-1])

## 11. Progress across rounds

In [ ]:
import csv
import matplotlib.pyplot as plt

print(f"{'round':>5} {'val once':>9} {'val end':>8} {'driver once (collect)':>22}")
for p in progress:
    driver = f"{p['on_policy']['success_once']:.0%}" if p["on_policy"] else "-"
    print(f"{p['round']:>5} {p['val_success_once']:>9.0%} {p['val_success_at_end']:>8.0%} {driver:>22}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
rounds = [p["round"] for p in progress]
ax1.plot(rounds, [p["val_success_once"] for p in progress], "o-", label="success once")
ax1.plot(rounds, [p["val_success_at_end"] for p in progress], "s-", label="success at end")
ax1.axhline(0.98, ls="--", c="gray", lw=1, label="PPO expert once (run 1)")
ax1.set(xlabel="DAgger round", ylabel="validation rate", ylim=(0, 1.02), title="Best checkpoint per round")
ax1.legend()
for r in rounds:
    rows = list(csv.DictReader((RUNS / f"round{r}" / "train_log.csv").open()))
    ax2.plot([int(x["iter"]) for x in rows], [float(x["val_mse"]) for x in rows], label=f"round {r}")
ax2.set(xlabel="iteration", ylabel="validation action MSE", yscale="log", title="Action error on held-out demos")
ax2.legend()
fig.tight_layout()
fig.savefig(RUNS / "dagger_progress.png", dpi=120)
plt.show()

## 12. Final test on fresh seeds

Picks the round with the best **validation** success (success-at-end, then success-once, later round on ties),
then evaluates it once on the test seeds next to the PPO expert. Saves a few videos.

In [ ]:
chosen = max(progress, key=lambda p: (p["val_success_at_end"], p["val_success_once"], p["round"]))
FINAL = RUNS / f"round{chosen['round']}" / "best.pt"
shutil.copy2(FINAL, RUNS / "teacher.pt")
print("Selected round", chosen["round"], FINAL)

run([TRAIN_PY, BC, "eval", "--checkpoint", FINAL, "--episodes", EVAL_EPISODES,
     "--seed", TEST_SEED, "--out", RUNS / "test_eval.json"])
run([TRAIN_PY, BC, "eval", "--checkpoint", EXPERT, "--released-ppo", "--episodes", EVAL_EPISODES,
     "--seed", TEST_SEED, "--out", OUTPUT_ROOT / "released_ppo_test_eval.json"])
run([TRAIN_PY, BC, "eval", "--checkpoint", FINAL, "--episodes", 4, "--seed", TEST_SEED,
     "--video-dir", RUNS / "videos"])

result = {"teacher": TEACHER_NAME, "selected_round": chosen["round"], "progress": progress,
          "test": json.loads((RUNS / "test_eval.json").read_text()),
          "released_ppo_test": json.loads((OUTPUT_ROOT / "released_ppo_test_eval.json").read_text()),
          "run1_test_for_comparison": {"success_once": 0.19, "success_at_end": 0.10}}
(RUNS / "summary.json").write_text(json.dumps(result, indent=2))
print(json.dumps({k: result[k] for k in ("selected_round", "test", "released_ppo_test", "run1_test_for_comparison")}, indent=2))
print("Everything is saved in", OUTPUT_ROOT)

## 13. Watch a rollout

In [ ]:
from IPython.display import Video, display
videos = sorted((RUNS / "videos").glob("*.mp4"))
print(len(videos), "videos")
if videos:
    display(Video(str(videos[0]), embed=True, width=480))